# Gradient boosting on log loss

A factory logs two sensor readings for every machine part it tests, and whether the part
failed. You will write gradient boosting for this yes-or-no label in two parts: the loop
that fits each tree to the pseudo-residuals of log loss, then the
validation curve that says how many rounds to keep.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

rng = np.random.default_rng(11)

def make_parts(n):
    """n parts: two sensor readings each, and 1 if the part failed."""
    X = rng.uniform(-2, 2, size=(n, 2))
    failed = rng.random(n) < sigmoid(2.0 * X[:, 0] * X[:, 1] - 0.8)
    return X, failed.astype(int)

# 400 parts to train on and 400 to validate on
X_train, y_train = make_parts(400)
X_val, y_val = make_parts(400)

print(f"{len(y_train)} training parts, {y_train.mean():.0%} of them faulty")

## Part 1 (coding task): The boosting loop

Write `fit_boosted(X, y, rounds, lr, depth)` for labels `y` of 0 and 1. The model's score
$F$ is a log-odds, and its probability is $p = \sigma(F)$. Start every row at the best
constant score $F_0$ for log loss, then repeat `rounds` times: compute each row's
pseudo-residual for log loss, fit a `DecisionTreeRegressor(max_depth=depth,
random_state=0)` to the pseudo-residuals, and step $F \leftarrow F + \text{lr} \cdot h$.
Return the pair `(f0, trees)`, with the trees in the order they were fit.

In [ ]:
def fit_boosted(X, y, rounds, lr=0.1, depth=2):
    """Return (f0, trees) for gradient boosting on log loss."""
    y = np.asarray(y, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
X8 = np.arange(8.0).reshape(-1, 1)
y8 = np.array([0, 1, 0, 0, 1, 1, 0, 0])
f0, trees = fit_boosted(X8, y8, rounds=3, lr=0.5, depth=2)
assert np.isclose(f0, np.log(3 / 5)), f"f0 = {f0}: with 3 positives in 8 the best constant score is the log-odds log(3/5), not the share 3/8"
assert len(trees) == 3, f"expected 3 trees, got {len(trees)}"
F = f0 + 0.5 * sum(t.predict(X8) for t in trees)
expected = [-0.677479, -0.677479, -0.677479, -0.677479, 0.316841, 0.316841, -1.012626, -1.012626]
assert np.allclose(F, expected, atol=1e-5), (
    f"scores after 3 rounds {np.round(F, 4)}, expected {expected}: fit each tree to y - p with "
    "p = sigmoid(F) recomputed every round, and step by lr times the tree's prediction")
print("Part 1 passed")

## Part 2 (coding task): Choosing the number of rounds

Write `staged_log_loss(model, X, y, lr)`. It returns an array whose entry `m - 1` is the
mean log loss on `(X, y)` of the model that stops after `m` rounds, for `m` from 1 to the
number of trees. Use the same `lr` the model was fit with, and clip the probabilities to
$[10^{-12}, 1 - 10^{-12}]$ before taking logs.

The check then boosts the training parts for 300 rounds at a learning rate of 0.3 with
trees of depth 3, and keeps the round where the validation loss is lowest.

In [ ]:
def staged_log_loss(model, X, y, lr=0.1):
    """Mean log loss after each round, one entry per tree."""
    f0, trees = model
    y = np.asarray(y, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
class Step:                       # a stand-in tree that adds the same amount to every row
    def __init__(self, value):
        self.value = value
    def predict(self, X):
        return np.full(len(X), self.value)

losses = staged_log_loss((0.0, [Step(1.0), Step(-0.5)]), np.zeros((3, 1)), [1, 0, 1], lr=0.5)
assert len(losses) == 2, f"expected one loss per round, 2 in all, got {len(losses)}"
assert np.allclose(losses, [0.640744, 0.659273], atol=1e-5), (
    f"losses {np.round(losses, 4)}, expected [0.6407, 0.6593]: the scores are 0.5 and then 0.25, "
    "and the loss uses the probabilities sigmoid(F)")

losses = staged_log_loss((0.0, [Step(50.0)]), np.zeros((2, 1)), [0, 0], lr=1.0)
assert np.all(np.isfinite(losses)), (
    f"losses {losses}: sigmoid(50) is exactly 1.0 in float64, so without the clip to "
    "1 - 1e-12 the log of (1 - p) is -inf on every row and the mean is inf")
assert losses[0] > 27.0, (
    f"loss {losses[0]:.4f}: with the clip, each row costs about -log(1e-12) = 27.63, so a "
    "loss near zero means the probabilities were never clipped")

losses = staged_log_loss((f0, trees), X8, y8, lr=0.5)
assert len(losses) == 3, f"expected one loss per round, 3 in all, got {len(losses)}"
assert np.allclose(losses, [0.595417, 0.544114, 0.504319], atol=1e-5), (
    f"losses {np.round(losses, 4)}, expected [0.5954, 0.5441, 0.5043]: turn the scores into "
    "probabilities with sigmoid before the log loss, and add lr times each tree")

model = fit_boosted(X_train, y_train, rounds=300, lr=0.3, depth=3)
train = staged_log_loss(model, X_train, y_train, lr=0.3)
val = staged_log_loss(model, X_val, y_val, lr=0.3)
best = int(val.argmin()) + 1
print(f"validation log loss is lowest after {best} rounds")
for m in (1, 10, best, 300):
    print(f"  after {m:>3} rounds: train {train[m - 1]:.3f}   validation {val[m - 1]:.3f}")
assert np.all(np.diff(train[:50]) < 0), "the training loss should fall every round early on"
assert 20 < best < 200 and val[-1] > val[best - 1] + 0.02, "expected the validation loss to turn up well before round 300"
print("Part 2 passed")

## Solution

### Part 1: The boosting loop

In [ ]:
def fit_boosted(X, y, rounds, lr=0.1, depth=2):
    """Return (f0, trees) for gradient boosting on log loss."""
    y = np.asarray(y, dtype=float)
    share = y.mean()
    f0 = np.log(share / (1 - share))          # the log-odds of the positive share
    F = np.full(len(y), f0)
    trees = []
    for _ in range(rounds):
        residual = y - sigmoid(F)              # -dL/dF for log loss
        tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X, residual)
        F += lr * tree.predict(X)
        trees.append(tree)
    return f0, trees

**Why it works.** For log loss with $p = \sigma(F)$, differentiating the loss in the score $F$
gives $p - y$, so the pseudo-residual is $y - \sigma(F)$, recomputed from the current
scores every round. Fitting to $y - F$, the squared-error residual, mixes a label with a
log-odds. The best constant solves $\sum_i (y_i - \sigma(F_0)) = 0$, so $\sigma(F_0)$ is
the share of positives and $F_0$ its log-odds. Starting at the share itself, 3/8, would
start from a probability of $\sigma(0.375) = 0.59$ instead of 0.375.

### Part 2: Choosing the number of rounds

In [ ]:
def staged_log_loss(model, X, y, lr=0.1):
    """Mean log loss after each round, one entry per tree."""
    f0, trees = model
    y = np.asarray(y, dtype=float)
    F = np.full(len(y), f0)
    losses = []
    for tree in trees:
        F = F + lr * tree.predict(X)
        p = np.clip(sigmoid(F), 1e-12, 1 - 1e-12)
        losses.append(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
    return np.array(losses)

**Why it works.** A model that stops after $m$ rounds is $F_0$ plus the first $m$ trees, each scaled
by the learning rate, so one pass that adds the trees in order gives every stage. The
training loss falls every round because each tree is a step downhill on it. The
validation loss falls while the trees fit the shape of the failure region and then rises
as they start fitting the noisy labels of individual training parts, so the round count
is a hyperparameter read off the validation curve. The minimum is itself picked from 300
noisy numbers, so a final score belongs on rows that played no part in choosing it.